# YOLO26s GAM - Object Size Analysis

This notebook evaluates `YOLO26s_GAM/weights/best.pt` on the test split using the same final baseline protocol: confidence 0.25, one-to-one class-aware matching at IoU 0.50, and relative-area size bins.

In [1]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO

PROJECT = Path(r"D:\\master\\Master_Drone_Detection")
MODEL_PATH = PROJECT / "04_experiments" / "YOLO26s_GAM" / "weights" / "best.pt"
GAM_MODULE_PATH = PROJECT / "03_code" / "attention" / "gam.py"
DATASET = PROJECT / "02_datasets" / "DUT_Anti_UAV"
TEST_IMAGES = DATASET / "images" / "test"
TEST_LABELS = DATASET / "labels" / "test"
OUTPUT_DIR = PROJECT / "04_experiments" / "YOLO26s_GAM_size_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONF_THRESHOLD = 0.25
MATCH_IOU_THRESHOLD = 0.50
IMAGE_SIZE = 960

assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}"
assert GAM_MODULE_PATH.exists(), f"Missing GAM module: {GAM_MODULE_PATH}"
assert TEST_IMAGES.exists(), f"Missing images: {TEST_IMAGES}"
assert TEST_LABELS.exists(), f"Missing labels: {TEST_LABELS}"

module_name = "ultralytics.nn.modules.gam"
spec = importlib.util.spec_from_file_location(module_name, GAM_MODULE_PATH)
gam_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = gam_module
spec.loader.exec_module(gam_module)

model = YOLO(str(MODEL_PATH))
print(f"Model: {MODEL_PATH}")
print(f"Test images: {len(list(TEST_IMAGES.glob('*')))}")

Model: D:\master\Master_Drone_Detection\04_experiments\YOLO26s_GAM\weights\best.pt
Test images: 2200


In [2]:
def yolo_to_xyxy(xc, yc, width, height, image_width, image_height):
    return [
        (xc - width / 2) * image_width,
        (yc - height / 2) * image_height,
        (xc + width / 2) * image_width,
        (yc + height / 2) * image_height,
    ]


def size_category(box, image_width, image_height):
    width = max(0.0, box[2] - box[0])
    height = max(0.0, box[3] - box[1])
    area_ratio = (width * height) / (image_width * image_height)

    if area_ratio < 0.0005:
        return "Tiny"
    if area_ratio < 0.002:
        return "Small"
    if area_ratio < 0.01:
        return "Medium"
    return "Large"


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0.0


def read_ground_truth(label_path, image_width, image_height):
    boxes = []
    if not label_path.exists():
        return boxes

    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls, xc, yc, width, height = map(float, parts[:5])
        box = yolo_to_xyxy(xc, yc, width, height, image_width, image_height)
        boxes.append({
            "class": int(cls),
            "box": box,
            "size": size_category(box, image_width, image_height),
        })
    return boxes


def match_boxes(gt_boxes, pred_boxes, iou_threshold):
    candidates = []
    for gt_index, gt in enumerate(gt_boxes):
        for pred_index, pred in enumerate(pred_boxes):
            if gt["class"] != pred["class"]:
                continue
            iou = calculate_iou(gt["box"], pred["box"])
            if iou >= iou_threshold:
                candidates.append((iou, gt_index, pred_index))

    matched_gt = set()
    matched_pred = set()
    for _, gt_index, pred_index in sorted(candidates, reverse=True):
        if gt_index not in matched_gt and pred_index not in matched_pred:
            matched_gt.add(gt_index)
            matched_pred.add(pred_index)
    return matched_gt

In [3]:
size_order = ["Tiny", "Small", "Medium", "Large"]
stats = {size: {"TP": 0, "FN": 0} for size in size_order}

results = model.predict(
    source=str(TEST_IMAGES),
    imgsz=IMAGE_SIZE,
    conf=CONF_THRESHOLD,
    device=0,
    save=False,
    verbose=False,
    stream=True,
)

for result in tqdm(results, total=len(list(TEST_IMAGES.glob('*'))), desc="Evaluating GAM"):
    image_height, image_width = result.orig_shape
    label_path = TEST_LABELS / f"{Path(result.path).stem}.txt"
    gt_boxes = read_ground_truth(label_path, image_width, image_height)

    pred_boxes = [
        {"class": int(cls), "box": box.tolist()}
        for cls, box in zip(
            result.boxes.cls.cpu().numpy(),
            result.boxes.xyxy.cpu().numpy(),
        )
    ]

    matched_gt = match_boxes(gt_boxes, pred_boxes, MATCH_IOU_THRESHOLD)
    for gt_index, gt in enumerate(gt_boxes):
        outcome = "TP" if gt_index in matched_gt else "FN"
        stats[gt["size"]][outcome] += 1

rows = []
for size in size_order:
    tp = stats[size]["TP"]
    fn = stats[size]["FN"]
    total_gt = tp + fn
    rows.append({
        "Size": size,
        "TP": tp,
        "FN": fn,
        "Total_GT": total_gt,
        "Recall": tp / total_gt if total_gt else 0.0,
    })

size_df = pd.DataFrame(rows)
output_path = OUTPUT_DIR / "size_analysis.csv"
size_df.to_csv(output_path, index=False)

print(size_df.to_string(index=False, formatters={"Recall": lambda value: f"{value:.6f}"}))
print(f"\nSaved: {output_path}")

  Size  TP  FN  Total_GT   Recall
  Tiny 812  24       836 0.971292
 Small 473  20       493 0.959432
Medium 354  20       374 0.946524
 Large 519  23       542 0.957565

Saved: D:\master\Master_Drone_Detection\04_experiments\YOLO26s_GAM_size_analysis\size_analysis.csv


In [5]:
from ultralytics import YOLO
from pathlib import Path

PROJECT = Path(r"D:\master\Master_Drone_Detection")

GAM_WEIGHTS = PROJECT / "04_experiments" / "YOLO26s_GAM" / "weights" / "best.pt"

model = YOLO(str(GAM_WEIGHTS))

In [6]:
TEST_IMAGES = PROJECT / "02_datasets" / "DUT_Anti_UAV" / "images" / "test"

results = model.predict(
    source=str(TEST_IMAGES),
    imgsz=960,
    conf=0.25,
    device=0,
    save=False,
    stream=True
)

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import cv2

from ultralytics import YOLO


PROJECT = Path(r"D:\master\Master_Drone_Detection")

MODEL_PATH = (
    PROJECT /
    "04_experiments" /
    "YOLO26s_GAM" /
    "weights" /
    "best.pt"
)

TEST_IMAGES = (
    PROJECT /
    "02_datasets" /
    "DUT_Anti_UAV" /
    "images" /
    "test"
)

TEST_LABELS = (
    PROJECT /
    "02_datasets" /
    "DUT_Anti_UAV" /
    "labels" /
    "test"
)


model = YOLO(str(MODEL_PATH))

In [8]:
def box_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0,x2-x1) * max(0,y2-y1)

    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter/union if union else 0

In [9]:
TP = 0
FP = 0
FN = 0

errors = {
    "Background_FP":0,
    "Localization_Error":0,
    "Duplicate":0
}


results = model.predict(
    source=str(TEST_IMAGES),
    imgsz=960,
    conf=0.25,
    device=0,
    stream=True,
    verbose=False
)


for r in tqdm(results):

    img_name = Path(r.path).stem

    label_file = TEST_LABELS / f"{img_name}.txt"


    if not label_file.exists():
        continue


    img = cv2.imread(r.path)

    h,w = img.shape[:2]


    # GT
    gt_boxes=[]

    with open(label_file) as f:
        for line in f:

            c,x,y,bw,bh = map(float,line.split())

            x1=(x-bw/2)*w
            y1=(y-bh/2)*h
            x2=(x+bw/2)*w
            y2=(y+bh/2)*h

            gt_boxes.append(
                [x1,y1,x2,y2]
            )


    preds=[]

    if r.boxes is not None:

        for b in r.boxes.xyxy.cpu().numpy():

            preds.append(list(b))


    matched=set()


    for p in preds:

        best_iou=0
        best_gt=-1

        for i,g in enumerate(gt_boxes):

            if i in matched:
                continue

            iou=box_iou(p,g)

            if iou>best_iou:
                best_iou=iou
                best_gt=i


        if best_iou>=0.5:

            TP+=1
            matched.add(best_gt)

        else:

            FP+=1

            if best_iou>0.1:
                errors["Localization_Error"]+=1
            else:
                errors["Background_FP"]+=1


    FN += len(gt_boxes)-len(matched)



print("TP:",TP)
print("FP:",FP)
print("FN:",FN)

print(errors)

2200it [01:46, 20.57it/s]

TP: 2158
FP: 79
FN: 87
{'Background_FP': 47, 'Localization_Error': 32, 'Duplicate': 0}


In [12]:
from ultralytics import YOLO

model = YOLO(
    r"D:\master\Master_Drone_Detection\05_models\pretrained\yolo26s_p2.yaml"
)

model.info()

YOLO26s_p2 summary: 329 layers, 9,765,856 parameters, 9,765,856 gradients, 28.0 GFLOPs


(329, 9765856, 9765856, 28.0226048)

In [13]:
for i, m in enumerate(model.model.model):
    print(i, m.__class__.__name__)

0 Conv
1 Conv
2 C3k2
3 Conv
4 C3k2
5 Conv
6 C3k2
7 Conv
8 C3k2
9 SPPF
10 C2PSA
11 Upsample
12 Concat
13 C3k2
14 Upsample
15 Concat
16 C3k2
17 Upsample
18 Concat
19 C3k2
20 Conv
21 Concat
22 C3k2
23 Conv
24 Concat
25 C3k2
26 Conv
27 Concat
28 C3k2
29 Detect
